# FallGuard Backend
### Instructions
1. Run **Cell 1** — installs packages
2. Run **Cell 2** — uploads model files
3. Run **Cell 3** — paste your Twilio token
4. Run **Cell 4** — starts server, prints your URL

> Wait for each cell to finish before running the next one

In [ ]:
# ── CELL 2: Upload model files ────────────────────────────────
import os
from google.colab import files

def upload_file(target_name, min_size=100):
    print(f'Select {target_name} when file picker opens...')
    uploaded = files.upload()
    if not uploaded:
        print(f'ERROR: No file selected for {target_name}')
        return False
    for name, data in uploaded.items():
        with open(target_name, 'wb') as f:
            f.write(data)
        size = os.path.getsize(target_name)
        print(f'Saved {target_name}: {size:,} bytes ({size//1024} KB) — OK')
        if size < min_size:
            print(f'WARNING: File looks empty — wrong file selected?')
            return False
        return True

print('='*40)
print('STEP 1: Upload model.pkl')
print('='*40)
ok1 = upload_file('model.pkl', min_size=100000)  # model is large ~16MB

print()
print('='*40)
print('STEP 2: Upload label_encoder.pkl')
print('='*40)
ok2 = upload_file('label_encoder.pkl', min_size=100)  # encoder is small, just check not empty

print()
if ok1 and ok2:
    print('Both files uploaded successfully — run Cell 3')
else:
    print('Something failed — re-run Cell 2')

In [ ]:
# ── CELL 2: Upload model files ────────────────────────────────
import os
from google.colab import files

def upload_file(target_name):
    print(f'Select {target_name} when file picker opens...')
    uploaded = files.upload()
    if not uploaded:
        print(f'ERROR: No file selected for {target_name}')
        return False
    # Write from memory in one shot — prevents corruption
    for name, data in uploaded.items():
        with open(target_name, 'wb') as f:
            f.write(data)
        size = os.path.getsize(target_name)
        print(f'Saved {target_name}: {size:,} bytes ({size//1024} KB)')
        if size < 1000:
            print(f'WARNING: File too small — likely wrong file selected')
            return False
        return True

# Upload model.pkl
print('='*40)
print('STEP 1: Upload model.pkl')
print('='*40)
ok1 = upload_file('model.pkl')

# Upload label_encoder.pkl
print()
print('='*40)
print('STEP 2: Upload label_encoder.pkl')
print('='*40)
ok2 = upload_file('label_encoder.pkl')

print()
if ok1 and ok2:
    print('Both files uploaded successfully!')
    print('Run Cell 3 next')
else:
    print('Some files failed — re-run Cell 2')

In [ ]:
# ── CELL 3: Twilio credentials ────────────────────────────────
# Sign up free: https://twilio.com/try-twilio
# IMPORTANT: Go to console.twilio.com and REGENERATE your Auth Token
# because the old one was exposed publicly

TWILIO_SID     = 'ACb64352faa1130520f248e1f03944609a'
TWILIO_TOKEN   = 'PASTE_NEW_TOKEN_HERE'   # regenerate at console.twilio.com
TWILIO_FROM    = '+12542744862'
EMERGENCY_TO   = '+919557064499'
EMERGENCY_NAME = 'Mom'
SOS_MIN_CONF   = 80    # % confidence needed to trigger SOS
SOS_COOLDOWN   = 60    # seconds between SMS alerts

# Validate
errors = []
if not TWILIO_SID.startswith('AC'):
    errors.append('TWILIO_SID looks wrong')
if TWILIO_TOKEN == 'PASTE_NEW_TOKEN_HERE':
    errors.append('You must paste your real Twilio token')
if not EMERGENCY_TO.startswith('+'):
    errors.append('EMERGENCY_TO must start with + and country code')

if errors:
    for e in errors:
        print(f'ERROR: {e}')
else:
    print(f'Credentials set')
    print(f'SOS will be sent to: {EMERGENCY_NAME} ({EMERGENCY_TO})')
    print('Run Cell 4 next')

In [ ]:
# ── CELL 4: Start backend + tunnel ───────────────────────────
import threading, time, joblib, pandas as pd
import subprocess, re, datetime, os
from flask import Flask, request, jsonify
from flask_cors import CORS
from twilio.rest import Client as TwilioClient

# ── Check files exist ────────────────────────────────────────
for fname in ['model.pkl', 'label_encoder.pkl']:
    if not os.path.exists(fname):
        raise FileNotFoundError(f'{fname} not found — run Cell 2 first')
    if os.path.getsize(fname) < 1000:
        raise ValueError(f'{fname} is corrupted — re-run Cell 2 and upload again')

# ── Load model ───────────────────────────────────────────────
print('Loading model...')
model = joblib.load('model.pkl')
le    = joblib.load('label_encoder.pkl')
print('Model loaded OK')

# ── Twilio ───────────────────────────────────────────────────
twilio_client = TwilioClient(TWILIO_SID, TWILIO_TOKEN)
print('Twilio client ready')

# ── Constants ────────────────────────────────────────────────
FEATURE_COLS = [
    'acc_max','gyro_max','acc_kurtosis','gyro_kurtosis',
    'lin_max','acc_skewness','gyro_skewness','post_gyro_max','post_lin_max'
]
LABEL_MAP = {
    'FOL':{'name':'Forward Fall','category':'fall'},
    'FKL':{'name':'Fall Kneel','category':'fall'},
    'SDL':{'name':'Sideways Fall','category':'fall'},
    'BSC':{'name':'Back Fall','category':'fall'},
    'STU':{'name':'Stumble','category':'stumble'},
    'WAL':{'name':'Walking','category':'normal'},
    'JOG':{'name':'Jogging','category':'normal'},
    'STD':{'name':'Standing','category':'normal'},
    'STN':{'name':'Standing Still','category':'normal'},
    'JUM':{'name':'Jumping','category':'normal'},
    'CSI':{'name':'Chair Sit-In','category':'normal'},
    'CSO':{'name':'Chair Sit-Out','category':'normal'},
    'SCH':{'name':'Sit Chair','category':'normal'},
}

last_sos_time = None

# ── SMS function ─────────────────────────────────────────────
def send_sms(label, confidence):
    global last_sos_time
    now = datetime.datetime.now()
    if last_sos_time:
        elapsed = (now - last_sos_time).seconds
        if elapsed < SOS_COOLDOWN:
            remaining = SOS_COOLDOWN - elapsed
            print(f'SMS cooldown: {remaining}s remaining')
            return {'sent': False, 'reason': 'cooldown', 'remaining': remaining}
    name = LABEL_MAP.get(label, {}).get('name', label)
    body = (
        f'FALL ALERT! {name} detected with {confidence}% confidence '
        f'at {now.strftime("%H:%M:%S")}. '
        f'Please check on {EMERGENCY_NAME} immediately!'
    )
    try:
        msg = twilio_client.messages.create(
            body=body, from_=TWILIO_FROM, to=EMERGENCY_TO
        )
        last_sos_time = now
        print(f'SMS SENT to {EMERGENCY_NAME} | SID: {msg.sid}')
        return {'sent': True, 'sid': msg.sid}
    except Exception as e:
        print(f'SMS FAILED: {e}')
        return {'sent': False, 'reason': str(e)}

# ── Flask routes ─────────────────────────────────────────────
app = Flask(__name__)
CORS(app)

@app.route('/')
def home():
    return jsonify({'status': 'FallGuard running', 'sos_to': EMERGENCY_TO})

@app.route('/api/predict', methods=['POST'])
def predict():
    try:
        data       = request.get_json(force=True)
        features   = {col: float(data.get(col, 0)) for col in FEATURE_COLS}
        df         = pd.DataFrame([features])
        pred       = model.predict(df)[0]
        proba      = model.predict_proba(df)[0]
        label      = le.inverse_transform([pred])[0]
        classes    = le.inverse_transform(list(range(len(proba))))
        top5       = sorted(zip(classes, proba), key=lambda x: -x[1])[:5]
        meta       = LABEL_MAP.get(label, {'name': label, 'category': 'unknown'})
        confidence = round(float(max(proba)) * 100, 1)
        return jsonify({
            'label':      label,
            'name':       meta['name'],
            'category':   meta['category'],
            'confidence': confidence,
            'top5': [
                {'label': l, 'name': LABEL_MAP.get(l, {}).get('name', l),
                 'prob': round(float(p) * 100, 1)}
                for l, p in top5
            ]
        })
    except Exception as e:
        import traceback
        return jsonify({'error': str(e), 'trace': traceback.format_exc()}), 500

@app.route('/api/sos', methods=['POST'])
def sos():
    try:
        data       = request.get_json(force=True)
        label      = data.get('label', 'FALL')
        confidence = data.get('confidence', 0)
        return jsonify(send_sms(label, confidence))
    except Exception as e:
        return jsonify({'sent': False, 'reason': str(e)}), 500

@app.route('/api/voice', methods=['POST'])
def voice_check():
    try:
        data     = request.get_json(force=True)
        response = data.get('response', '').lower().strip()
        label    = data.get('label', 'FALL')
        conf     = data.get('confidence', 0)
        ok_words  = ['ok','okay','fine','i am ok','i am okay','im ok',
                     'im fine','no','cancel','safe','haan','theek']
        sos_words = ['help','yes','send','sos','call','emergency',
                     'fallen','hurt','pain','madad','nahi']
        is_ok  = any(w in response for w in ok_words)
        is_sos = any(w in response for w in sos_words)
        if is_ok:
            print(f'Voice: OK — "{response}"')
            return jsonify({'action': 'cancel', 'heard': response})
        elif is_sos:
            print(f'Voice: SOS — "{response}"')
            result = send_sms(label, conf)
            result['action'] = 'sos'
            result['heard']  = response
            return jsonify(result)
        else:
            print(f'Voice: unclear — "{response}"')
            return jsonify({'action': 'unclear', 'heard': response})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/predict-csv', methods=['POST'])
def predict_csv():
    try:
        if 'file' not in request.files:
            return jsonify({'error': 'No file uploaded'}), 400
        df      = pd.read_csv(request.files['file'])
        df      = df.drop(columns=['Unnamed: 0','label','fall'], errors='ignore')
        missing = [c for c in FEATURE_COLS if c not in df.columns]
        if missing:
            return jsonify({'error': f'Missing columns: {missing}'}), 400
        preds   = model.predict(df[FEATURE_COLS])
        labels  = le.inverse_transform(preds)
        results, counts = [], {}
        for i, lbl in enumerate(labels):
            m = LABEL_MAP.get(lbl, {'name': lbl, 'category': 'unknown'})
            results.append({'row': i+1, 'label': lbl,
                            'name': m['name'], 'category': m['category']})
            counts[m['category']] = counts.get(m['category'], 0) + 1
        return jsonify({'total': len(results), 'results': results, 'summary': counts})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/stats')
def stats():
    return jsonify({'message': 'Stats not available in Colab mode'})

# ── Start Flask ──────────────────────────────────────────────
flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False)
)
flask_thread.daemon = True
flask_thread.start()
time.sleep(3)
print('Flask running on port 5000')

# ── Cloudflare tunnel ────────────────────────────────────────
print('Starting tunnel...')
os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared')

tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

url = None
for _ in range(40):
    line = tunnel.stderr.readline().decode('utf-8', errors='ignore')
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print()
if url:
    print('=' * 60)
    print('  BACKEND IS LIVE!')
    print('=' * 60)
    print()
    print('  Paste this URL in your frontend:')
    print(f'  {url}/api/predict')
    print()
    print(f'  SOS SMS will go to: {EMERGENCY_NAME} ({EMERGENCY_TO})')
    print()
    print('  Keep this tab open!')
    print('=' * 60)
else:
    print('Tunnel failed — try running Cell 4 again')